In [3]:
import os
import json
import urllib.request
import urllib.parse
import csv

BASE_DIR = "data"
SRP_ID = "SRP151023"
RUN_TABLE = f"{BASE_DIR}/SraRunTable.csv"

# 1. Download the SraRunTable immediately when Snakemake parses the file
os.makedirs(BASE_DIR, exist_ok=True)

if not os.path.exists(RUN_TABLE):
    print(f"SraRunTable.csv not found. Fetching from NCBI for {SRP_ID}...")
    
    search_params = {"db": "sra", "term": SRP_ID, "retmode": "json", "retmax": 500}
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?" + urllib.parse.urlencode(search_params)
    
    try:
        # Search SRA
        with urllib.request.urlopen(search_url) as response:
            id_list = json.loads(response.read().decode()).get("esearchresult", {}).get("idlist", [])
            
        if id_list:
            # Fetch Metadata
            fetch_params = {"db": "sra", "id": ",".join(id_list), "rettype": "runinfo", "retmode": "text"}
            fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?" + urllib.parse.urlencode(fetch_params)
            urllib.request.urlretrieve(fetch_url, RUN_TABLE)
            print("SraRunTable.csv downloaded successfully.")
    except Exception as e:
        print(f"Warning: Failed to retrieve SRA metadata: {e}")


SraRunTable.csv not found. Fetching from NCBI for SRP151023...
SraRunTable.csv downloaded successfully.


In [7]:
# 1. Parse SraRunTable to dynamically establish SRR (Run) -> GSM (GEO Sample) mapping
# Dynamically map GSM -> list of R1 SRRs and GSM -> list of R2 SRRs based on spot length
gsm_to_download = {}
srrs_to_download = []

if os.path.exists(RUN_TABLE):
    with open(RUN_TABLE, 'r') as f:
        first_line = f.readline()
        delimiter = '\t' if '\t' in first_line else ','
    with open(RUN_TABLE, 'r') as f:
        reader = csv.DictReader(f, delimiter=delimiter)
        headers = reader.fieldnames
        
        run_col = next((h for h in headers if h.lower() in ['run', 'run_s']), None)
        gsm_col = next((h for h in headers if h.lower() in ['samplename']), None)
        
        
        if run_col and gsm_col:
            for row in reader:
                srr = row[run_col].strip()
                gsm = row[gsm_col].strip()
                
                if srr and gsm:
                     if gsm not in gsm_to_download:
                        gsm_to_download[gsm] = []
                     gsm_to_download[gsm].append(srr)
                     srrs_to_download.append(srr)
                    # Note: spot_len == 8 (Index reads) are ignored entirely

GSMS = sorted(list(set(gsm_to_download.keys())))
SRRS = sorted(list(set(srrs_to_download)))

print(f"Identified {len(GSMS)} GSMs with both R1 and R2 SRRs.")
print(f"Identified {len(SRRS)} unique SRRs to download.")

Identified 10 GSMs with both R1 and R2 SRRs.
Identified 38 unique SRRs to download.
